# Using Ray with TimeCopilot

In [1]:

import nest_asyncio

nest_asyncio.apply()

from timecopilot import TimeCopilotForecaster

import ray 

from timecopilot.models import SeasonalNaive
from timecopilot.models.foundation.chronos import Chronos

2026-03-18 17:41:44,916	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.


## Create the dataframe

In [ ]:
df = ray.data.read_csv("https://timecopilot.s3.amazonaws.com/public/data/air_passengers.csv")

2026-03-18 17:41:47,154	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-03-18 17:41:48,526	INFO worker.py:1852 -- Started a local Ray instance.


                                                                                                                
                                                                                                                                               

                                                                                                                                        


                                                   



                                                




                                                





Running Dataset. Active & requested resources: 3/14 CPU, 768.0MB/1.0GB object store: : 0.00 row [00:05, ? row/s]                      
















(MapBatches(_udf) pid=72325)  See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.


                                                                                                               
                                                                                                                                               

                                                                                                                                        


                                                   



                                                




                                                





  0%|          | 0/1 [00:00<?, ?it/s]                                                                               (MapBatches(_udf) pid=72325) 
Running Dataset. Active & requested resources: 12/14 CPU, 5.2KB/1.0GB object store: : 0.00 row [00:07, ? row/s]














                                                                                                               
                                                          

(MapBatches(_udf) pid=72323)  See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs. [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)


In [14]:
df.show(5)

2026-03-18 17:46:56,978	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2026-03-18_17-41-47_368791_62158/logs/ray-data
2026-03-18 17:46:56,978	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadCSV] -> LimitOperator[limit=5]
Running 0: 0.00 row [00:00, ? row/s]
                                                                              
                                                    

✔️  Dataset execution finished in 0.17 seconds: : 5.00 row [00:00, 28.9 row/s]


                                                                                                                           

- ReadCSV->SplitBlocks(28): Tasks: 1; Queued blocks: 0; Resources: 1.0 CPU, 696.0B object store: : 18.0 row [00:00, 103 row/s]






- limit=5: Tasks: 0; Queued blocks: 2; Resources: 0.0 CPU, 145.0B object store: : 5.00 row [00:00, 28.5 row/s]

{'unique_id': 'AirPassengers', 'ds': datetime.date(1949, 1, 1), 'y': 112}
{'unique_id': 'AirPassengers', 'ds': datetime.date(1949, 2, 1), 'y': 118}
{'unique_id': 'AirPassengers', 'ds': datetime.date(1949, 3, 1), 'y': 132}
{'unique_id': 'AirPassengers', 'ds': datetime.date(1949, 4, 1), 'y': 129}
{'unique_id': 'AirPassengers', 'ds': datetime.date(1949, 5, 1), 'y': 121}


## Create a TimeCopilotForecaster

In [4]:
tcf = TimeCopilotForecaster(
    models=[
        SeasonalNaive(),
        Chronos("autogluon/chronos-2-small")
    ]
)

## Create a forecast

In [5]:
result = tcf.forecast(
    df=df,
    h=12
)

2026-03-18 17:42:29,431	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2026-03-18_17-41-47_368791_62158/logs/ray-data
2026-03-18 17:42:29,432	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadCSV]
                                                                                                 
✔️  Dataset execution finished in 0.63 seconds: 100%|██████████| 144/144 [00:00<00:00, 232 row/s]

- ReadCSV->SplitBlocks(28): Tasks: 0; Queued blocks: 0; Resources: 0.0 CPU, 290.0B object store: : 144 row [00:00, 301 row/s]
2026-03-18 17:42:30,110	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2026-03-18_17-41-47_368791_62158/logs/ray-data
2026-03-18 17:42:30,110	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadCSV]
                                                         

## View Forecast Results

In [6]:
result.to_pandas()

,unique_id,ds,SeasonalNaive,Chronos
0,AirPassengers,1961-01-01,417.0,454.0
1,AirPassengers,1961-02-01,391.0,430.0
2,AirPassengers,1961-03-01,419.0,506.0
3,AirPassengers,1961-04-01,461.0,504.0
4,AirPassengers,1961-05-01,472.0,512.0
5,AirPassengers,1961-06-01,535.0,584.0
6,AirPassengers,1961-07-01,622.0,660.0
7,AirPassengers,1961-08-01,606.0,668.0
8,AirPassengers,1961-09-01,508.0,568.0
9,AirPassengers,1961-10-01,461.0,488.0
